# ARIA-LLM — Fine-tune from Qwen2.5-0.5B on Google Colab (T4)

Full fp32 fine-tuning of the 358M-param model needs ~5.7 GB (weights + grads + AdamW state) plus activations — too much for a 4 GB laptop GPU, but fits on Colab's **16 GB T4** at a modest batch size.

This notebook: clone repo → rebuild the corpus with the **anti-collapse** flags → fine-tune → slim the checkpoint for deploy → (optional) upload weights to your Hugging Face model repo so the live Space can serve them.

**Before running:** `Runtime → Change runtime type → T4 GPU`, and push your repo to GitHub first (this notebook clones it).

In [ ]:
# 1. Confirm we have a GPU (expect: Tesla T4, ~15 GB)
!nvidia-smi

In [ ]:
# 2. Clone the repo (public). Change BRANCH if you pushed to a different one.
REPO = 'https://github.com/Aashutosh2021/ARIA_LLM.git'
BRANCH = 'main'

import os, shutil
if os.path.isdir('ARIA_LLM'):
    shutil.rmtree('ARIA_LLM')  # fresh clone so re-runs pick up new pushes
!git clone --branch $BRANCH --depth 1 $REPO
%cd ARIA_LLM

In [ ]:
# 3. Install only what Colab lacks (keep Colab's CUDA torch).
!pip install -q 'transformers>=4.40' 'PyYAML>=6.0' 'tqdm>=4.65' 'huggingface_hub>=0.24'

In [ ]:
# 4. Rebuild the corpus with anti-collapse settings. --max-answer-repeats caps
#    how often any single bot answer may appear (stops one canned reply from
#    dominating and collapsing the model, e.g. the old '456 cherry drive' bug).
#    Watch the 'Most common bot answers' report it prints — no answer should be
#    a large fraction of the total. This OVERWRITES data/chat_corpus.txt.
!python scripts/build_chat_corpus.py --max-answer-repeats 3 --clean-upsample 3

In [ ]:
# 5. Download + convert Qwen weights -> checkpoints/qwen2.5-0.5b-instruct.pt
!python scripts/import_qwen.py

In [ ]:
# 6. Fine-tune. This attention implementation keeps the full
#    (batch, heads, seq, seq) scores for backward, so memory scales with
#    batch_size * seq_len^2 -- batch 2 fits the T4 with headroom. If it OOMs,
#    drop to --seq-len 128.
!python train_qwen_finetune.py --device cuda --epochs 5 --batch-size 2 --seq-len 256

In [ ]:
# 7. Quick sanity check across several prompts — replies should VARY (not the
#    same canned answer to everything). Higher repetition_penalty (1.3) also
#    helps avoid loops on CPU/GPU alike.
import torch
from transformers import AutoTokenizer
from chat_qwen import load_model, generate
from utils.device import get_device

device = get_device('cuda')
model, config = load_model('checkpoints_chat_qwen/best.pt', device)
tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
tok.add_special_tokens({'additional_special_tokens': [config['user_token'], config['bot_token'], config['end_token']]})
stop_id = tok.convert_tokens_to_ids(config['end_token'])

for q in ['hello, who are you?', 'what can you do?', 'what is your name?', 'how are you?']:
    prompt = f"{config['user_token']} {q}\n{config['bot_token']} "
    ids = tok.encode(prompt, return_tensors='pt').to(device)
    out = [t for t in generate(model, ids, device, max_new_tokens=64, temperature=0.7,
                               top_p=0.9, repetition_penalty=1.3, max_context=512, stop_token_id=stop_id)]
    print(f'you> {q}\naria> {tok.decode(out, skip_special_tokens=True)}\n')

In [ ]:
# 8. Slim both checkpoints for deployment (drops optimizer state, ~6GB -> ~1.4GB).
!python scripts/export_for_deploy.py --checkpoint checkpoints_chat_qwen/best.pt --out deploy/aria_finetuned.pt
!python scripts/export_for_deploy.py --checkpoint checkpoints/qwen2.5-0.5b-instruct.pt --out deploy/aria_qwen.pt

## 9. Upload weights to your Hugging Face model repo

Needed so the live Space (see `DEPLOY.md`) can download them. Get a **write** token at https://huggingface.co/settings/tokens . Replace `<user>` below with your HF username.

In [ ]:
from huggingface_hub import login, create_repo, upload_file

HF_USER = '<user>'          # <-- your Hugging Face username
REPO_ID = f'{HF_USER}/aria-llm-weights'

login()  # paste your write token when prompted
create_repo(REPO_ID, repo_type='model', exist_ok=True)

for fname in ['aria_qwen.pt', 'aria_finetuned.pt']:
    upload_file(path_or_fileobj=f'deploy/{fname}', path_in_repo=fname,
                repo_id=REPO_ID, repo_type='model')
    print('uploaded', fname)
print('\nDone. Set HF_MODEL_REPO =', REPO_ID, 'in your Space settings.')